In [ ]:
# Install required packages
%pip install ultralytics --quiet

In [2]:
# Verify GPU is available
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU device     : {torch.cuda.get_device_name(0)
                          if torch.cuda.is_available() else 'None'}")

CUDA available : True
GPU device     : Tesla T4


In [1]:
!git clone https://github.com/NaveenSa98/construction-safety-monitor.git
%cd construction-safety-monitor

Cloning into 'construction-safety-monitor'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 69 (delta 26), reused 57 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 27.47 KiB | 6.87 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/construction-safety-monitor


In [ ]:
# Extract custom dataset from Google Drive into data/raw
import zipfile
import os
import shutil

# Update this path to match where you saved the zip in your Drive
DRIVE_ZIP = "/content/drive/MyDrive/construction_safety_dataset.zip"
RAW_DIR   = "data/raw"

os.makedirs(RAW_DIR, exist_ok=True)

print(f"[INFO] Extracting {DRIVE_ZIP} → {RAW_DIR} ...")
with zipfile.ZipFile(DRIVE_ZIP, "r") as z:
    z.extractall(RAW_DIR)
print("[INFO] Extraction complete.")

# Fix nested folder if zip extracted into data/raw/raw/
nested = os.path.join(RAW_DIR, "raw")
if os.path.isdir(nested):
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), os.path.join(RAW_DIR, item))
    os.rmdir(nested)
    print("[INFO] Flattened nested 'raw/' folder.")

# Verify raw structure
for split in ["train", "val", "test"]:
    imgs = len(os.listdir(f"{RAW_DIR}/images/{split}"))
    lbls = len(os.listdir(f"{RAW_DIR}/labels/{split}"))
    print(f"  {split:5s} — images: {imgs:4d} | labels: {lbls:4d}")

In [ ]:
# Run data preparation pipeline (shuffle + re-split raw → processed)
!python src/data_preparation.py

# Verify processed output
import os
for split in ["train", "val", "test"]:
    img_count = len(os.listdir(f"data/processed/images/{split}"))
    lbl_count = len(os.listdir(f"data/processed/labels/{split}"))
    print(f"{split:5s} — images: {img_count:4d} | labels: {lbl_count:4d}")

In [ ]:
# Confirm dataset.yaml is correct before training
import yaml

with open("data/processed/dataset.yaml", "r") as f:
    content = yaml.safe_load(f)

print("dataset.yaml contents:")
for key, value in content.items():
    print(f"  {key}: {value}")

# Sanity check: expect 12 classes
assert content["nc"] == 12, f"Expected 12 classes, got {content['nc']}"
expected = [
    "Person", "Hardhat", "Safety Vest", "Safety Boots", "Safety Gloves",
    "Safety Goggles", "NO-Hardhat", "NO-Safety Vest", "NO-Safety Boots",
    "NO-Safety Gloves", "NO-Safety Goggles", "Mask"
]
assert content["names"] == expected, "Class names mismatch!"
print("\n[OK] 12 classes verified.")

In [10]:
# Launch YOLOv8 fine-tuning
!python src/train.py

[INFO] Initialising YOLOv8 model...
[INFO] Pretrained weights loaded : yolov8m.pt
[INFO] Dataset                   : /content/construction-safety-monitor/data/processed/dataset.yaml
[INFO] Epochs                    : 50
[INFO] Image size                : 640
[INFO] Batch size                : 24
[INFO] Output directory          : /content/construction-safety-monitor/outputs/training_runs/ppe_detection_v1
[INFO] Starting training...

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/construction-safety-monitor/data/processed/dataset.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, era

In [ ]:
# Display training results
from IPython.display import Image, display
import os

results_dir = "outputs/training_runs/ppe_detection_v2"

print(f"Files in {results_dir}:")
for file in os.listdir(results_dir):
    print(f"  - {file}")

# Training curves
display(Image(f"{results_dir}/results.png"))

# Confusion matrix
display(Image(f"{results_dir}/confusion_matrix.png"))

# F1 curve
display(Image(f"{results_dir}/BoxF1_curve.png"))